In [19]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "dbrepo"], capture_output=True)

# Verify it installed
import importlib
print("dbrepo installed:", importlib.util.find_spec("dbrepo") is not None)

dbrepo installed: True


In [20]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTableColumn, ColumnType, CreateTableConstraints

# Connect to DBRepo
client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12534814",
    password="Aamivishnu@0914"
)

print("Connected successfully")

Connected successfully


In [21]:
# Delete all existing tables (clean start)
tables = client.get_tables(database_id=DATABASE_ID)
for table in tables:
    client.delete_table(database_id=DATABASE_ID, table_id=table.id)
    print(f"Deleted: {table.name}")
print("All old tables deleted")

Deleted: water_quality_measurement
Deleted: sampling_event
Deleted: sampling_station
Deleted: lake
All old tables deleted


In [22]:
DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"

db = client.get_database(database_id=DATABASE_ID)
print("Database found:", db.name)

Database found: lake_water_quality


In [23]:
import pandas as pd

# Load your CSV file
df = pd.read_csv(r"C:\Users\anusr\OneDrive\Desktop\Data Stewerdship\Lakes_Monitoring.csv")
print("CSV loaded:", df.shape)
print(df.columns[:5].tolist())

CSV loaded: (1935, 116)
['_type', '_id', '_revision', '_page.next', 'regionas']


In [24]:
# Remove system/admin columns
df_clean = df.drop(columns=['_type', '_id', '_revision', '_page.next'])
print("Clean shape:", df_clean.shape)

Clean shape: (1935, 112)


In [26]:
import numpy as np

def replace_below_detection(value):
    # Handle NaN/None
    if value is None:
        return np.nan
    if isinstance(value, float):
        return value
    # Convert to string and clean
    val_str = str(value).strip()
    # Handle empty or dash values
    if val_str in ['', '-', 'nan', 'None', 'Nematuota', 'nematuota']:
        return np.nan
    # Handle below detection limit <0,005
    if val_str.startswith('<'):
        num_str = val_str.replace('<', '').replace(',', '.').strip()
        try:
            return float(num_str) / 2
        except:
            return np.nan
    # Handle normal numbers with comma decimal
    try:
        return float(val_str.replace(',', '.'))
    except:
        return np.nan

# Apply fix to all pollutant columns
cols_to_fix = [
    'suspend_medziagos', 'sarmingumas', 'skaidrumas',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'fosfatu_fosforas', 'fosforas_bendras',
    'chlorofilas_a', 'gyvsidabris', 'kadmis', 'nikelis', 'svinas',
    'varis', 'chromas', 'vanadis', 'aliuminis', 'alavas', 'arsenas',
    'cinkas', 'antracenas', 'fluorantenas', 'naftalenas',
    'benz_a_pirenas', 'benz_b_fluorantenas', 'benz_k_fluorantenas',
    'benz_ghi_perilenas', 'inden_123_cd_pirenas', 'p4_n_nonilfenolis',
    'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot', 'p4_tert_oktilfenolis',
    'nonilfenoliai', 'pentachlorfenolis', 'benzenas', 'p12_dichloretanas',
    'p123_trichlorbenzenas', 'p124_trichlorbenzenas', 'heksachlorbutadienas',
    'trichloretilenas', 'tetrachlormetanas', 'dichlormetanas',
    'tetrachloretilenas', 'trichlormetanas', 'aldrinas', 'dieldrinas',
    'izodrinas', 'endrinas', 'alfa_heksachlorcikloheksanas',
    'beta_heksachlorcikloheksanas', 'gama_heksachlorcikloheksanas',
    'heksachlorbenzenas', 'pentachlorbenzenas', 'alfa_endosulfanas',
    'beta_endosulfanas', 'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde',
    'simazinas', 'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]

for col in cols_to_fix:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(replace_below_detection)

print("Fix applied successfully!")
print("Sample nitritu_azotas:", df_clean['nitritu_azotas'].head(3).tolist())
print("dtype:", df_clean['nitritu_azotas'].dtype)

Fix applied successfully!
Sample nitritu_azotas: [0.0025, 0.002, 0.003]
dtype: float64


In [27]:
#creating lake table
# Set lake_id as the index (primary key)
df_lake = df_clean[['telkinio_pav', 'regionas']].drop_duplicates()
df_lake.columns = ['lake_name', 'region']
df_lake = df_lake.reset_index(drop=True)
df_lake.index.name = "lake_id"

table_lake = client.create_table(
    database_id=DATABASE_ID,
    name="lake",
    is_public=True,
    is_schema_public=True,
    dataframe=df_lake,
    description="Unique lakes with their administrative region in Lithuania"
)
print("lake table created! Rows:", len(df_lake))

2026-05-23 02:58:24,904 root         WARNING default to 'text' for column lake_name and type <class 'numpy.dtype'>
2026-05-23 02:58:24,908 root         WARNING default to 'text' for column region and type <class 'numpy.dtype'>
lake table created! Rows: 346


In [28]:

# Prepare station data
df_station = df_clean[['m_vietos_kodas', 'm_vietos_pav', 'koord']].drop_duplicates()
df_station.columns = ['station_code', 'station_name', 'coordinates']
df_station = df_station.reset_index(drop=True)
df_station.index.name = "station_id"

table_station = client.create_table(
    database_id=DATABASE_ID,
    name="sampling_station",
    is_public=True,
    is_schema_public=True,
    dataframe=df_station,
    description="Monitoring stations with their official code and coordinates"
)
print("sampling_station table created! Rows:", len(df_station))

2026-05-23 02:58:56,002 root         WARNING default to 'text' for column station_code and type <class 'numpy.dtype'>
2026-05-23 02:58:56,004 root         WARNING default to 'text' for column station_name and type <class 'numpy.dtype'>
2026-05-23 02:58:56,007 root         WARNING default to 'text' for column coordinates and type <class 'numpy.dtype'>
sampling_station table created! Rows: 355


In [29]:
#Create sampling_station table
df_event = df_clean[['m_vietos_kodas', 'data']].copy()
df_event.columns = ['station_code', 'sampled_on']
df_event = df_event.reset_index(drop=True)
df_event.index.name = "event_id"

table_event = client.create_table(
    database_id=DATABASE_ID,
    name="sampling_event",
    is_public=True,
    is_schema_public=True,
    dataframe=df_event,
    description="Sampling visits - one row per station per date"
)
print("sampling_event table created! Rows:", len(df_event))


2026-05-23 02:59:36,802 root         WARNING default to 'text' for column station_code and type <class 'numpy.dtype'>
2026-05-23 02:59:36,806 root         WARNING default to 'text' for column sampled_on and type <class 'numpy.dtype'>
sampling_event table created! Rows: 1935


In [30]:
measurement_cols = [
    'vandens_temp', 'suspend_medziagos', 'sarmingumas',
    'deguonis_istirpes', 'ph', 'skaidrumas', 'elektr_laidis',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'azotas_mineralinis', 'azotas_bendras',
    'fosfatu_fosforas', 'fosforas_bendras', 'anglingumas',
    'chlorofilas_a', 'kalcio_karbonatas', 'gyvsidabris', 'kadmis',
    'nikelis', 'svinas', 'varis', 'chromas', 'vanadis', 'aliuminis',
    'alavas', 'arsenas', 'cinkas', 'antracenas', 'fluorantenas',
    'naftalenas', 'benz_a_pirenas', 'benz_b_fluorantenas',
    'benz_k_fluorantenas', 'benz_ghi_perilenas', 'inden_123_cd_pirenas',
    'p4_n_nonilfenolis', 'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot',
    'p4_tert_oktilfenolis', 'nonilfenoliai', 'pentachlorfenolis',
    'benzenas', 'p12_dichloretanas', 'p123_trichlorbenzenas',
    'p124_trichlorbenzenas', 'heksachlorbutadienas', 'trichloretilenas',
    'tetrachlormetanas', 'dichlormetanas', 'tetrachloretilenas',
    'trichlormetanas', 'aldrinas', 'dieldrinas', 'izodrinas', 'endrinas',
    'alfa_heksachlorcikloheksanas', 'beta_heksachlorcikloheksanas',
    'gama_heksachlorcikloheksanas', 'heksachlorbenzenas',
    'pentachlorbenzenas', 'alfa_endosulfanas', 'beta_endosulfanas',
    'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde', 'simazinas',
    'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]

df_measurement = df_clean[measurement_cols].copy()
df_measurement.index.name = "measurement_id"

table_meas = client.create_table(
    database_id=DATABASE_ID,
    name="water_quality_measurement",
    is_public=True,
    is_schema_public=True,
    dataframe=df_measurement,
    description="All physicochemical and pollutant measurements for each sampling event"
)
print("water_quality_measurement table created! Rows:", len(df_measurement))

water_quality_measurement table created! Rows: 1935


In [16]:
# Prepare measurement data — all chemical columns
measurement_cols = [
    'vandens_temp', 'suspend_medziagos', 'sarmingumas',
    'deguonis_istirpes', 'ph', 'skaidrumas', 'elektr_laidis',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'azotas_mineralinis', 'azotas_bendras',
    'fosfatu_fosforas', 'fosforas_bendras', 'anglingumas',
    'chlorofilas_a', 'kalcio_karbonatas', 'gyvsidabris', 'kadmis',
    'nikelis', 'svinas', 'varis', 'chromas', 'vanadis', 'aliuminis',
    'alavas', 'arsenas', 'cinkas', 'antracenas', 'fluorantenas',
    'naftalenas', 'benz_a_pirenas', 'benz_b_fluorantenas',
    'benz_k_fluorantenas', 'benz_ghi_perilenas', 'inden_123_cd_pirenas',
    'p4_n_nonilfenolis', 'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot',
    'p4_tert_oktilfenolis', 'nonilfenoliai', 'pentachlorfenolis',
    'benzenas', 'p12_dichloretanas', 'p123_trichlorbenzenas',
    'p124_trichlorbenzenas', 'heksachlorbutadienas', 'trichloretilenas',
    'tetrachlormetanas', 'dichlormetanas', 'tetrachloretilenas',
    'trichlormetanas', 'aldrinas', 'dieldrinas', 'izodrinas', 'endrinas',
    'alfa_heksachlorcikloheksanas', 'beta_heksachlorcikloheksanas',
    'gama_heksachlorcikloheksanas', 'heksachlorbenzenas',
    'pentachlorbenzenas', 'alfa_endosulfanas', 'beta_endosulfanas',
    'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde', 'simazinas',
    'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]

df_measurement = df_clean[measurement_cols].copy()
df_measurement.index.name = "measurement_id"

print("Measurements to insert:", len(df_measurement))
print("Columns:", len(df_measurement.columns))

Measurements to insert: 1935
Columns: 106


In [32]:
# Final check
tables = client.get_tables(database_id=DATABASE_ID)
print("All tables in DBRepo:")
for t in tables:
    print(f"  - {t.name}")

print()
print("DBRepo URL:")
print(f"https://test.dbrepo.tuwien.ac.at/database/{DATABASE_ID}/table")

All tables in DBRepo:
  - water_quality_measurement
  - sampling_event
  - sampling_station
  - lake

DBRepo URL:
https://test.dbrepo.tuwien.ac.at/database/13457a52-37f9-48d4-a078-6865e8d35981/table
